# Proyecto 1: EDA + Baseline

**Ciencia de Datos, Sección A** · Segundo semestre 2026

Entrega: **viernes 11 de septiembre** · 10 puntos

---

**Autor(es):** Philip Falla

**Dataset elegido:** Predict Students' Dropout and Academic Success (UCI Machine Learning Repository, dataset ID 697)

### Rúbrica resumida

| Criterio | Pts | Qué buscamos |
|---|---|---|
| Pregunta y datos | 2 | Pregunta clara, provenance del dataset |
| EDA | 3 | Hallazgos, no galería de gráficas |
| Features | 2 | Cada decisión justificada, sin leakage |
| Baseline | 2 | Validación correcta, métrica adecuada |
| Límites | 1 | Qué no puede concluirse con estos datos |

## 1. Pregunta y contexto

Las instituciones de educación superior suelen identificar tarde a los estudiantes en riesgo de abandonar: para cuando las notas de fin de semestre confirman el problema, ya pasó una cohorte entera de matrícula sin intervención. Si una universidad pudiera estimar ese riesgo usando solo la información que ya tiene el día en que un estudiante se inscribe —antes de que exista un solo registro de desempeño académico—, podría dirigir tutorías, becas o seguimiento a quienes más lo necesitan desde el primer semestre, en lugar de esperar a que el abandono ya esté en curso.

Este proyecto pregunta qué tanto explica esa información personal y socioeconómica —sin incluir desempeño curricular ni datos de admisión— la probabilidad de que un estudiante abandone la carrera. La respuesta le importa a una oficina de bienestar o retención estudiantil: si un modelo entrenado solo con este tipo de variables apenas supera predecir siempre "no abandona", la conclusión práctica es que el perfil personal no basta y hay que esperar señales académicas tempranas (primer parcial, asistencia) para intervenir con criterio. Si en cambio distingue con margen claro sobre ese baseline trivial, se justifica invertir en programas de apoyo dirigidos desde el momento de la admisión.

**Pregunta:** ¿Qué tanto explica la información personal y socioeconómica de un estudiante (género, edad, estado civil, nacionalidad, condición de desplazado, necesidades educativas especiales, educación y ocupación de los padres, si es deudor, si tiene las cuotas al día, si es becado, si es estudiante internacional) su probabilidad de abandonar la carrera, frente a simplemente predecir siempre la clase mayoritaria?

**Tipo de problema:** Clasificación binaria.

**Variable objetivo:** `Target`, recodificada como `dropout` (1 = Dropout, 0 = Enrolled o Graduate). Se colapsan "Enrolled" y "Graduate" en una sola clase de "no abandono" porque la pregunta es específicamente sobre abandono, no sobre distinguir entre seguir inscrito y graduarse.

## 2. Los datos

- **Fuente:** UCI Machine Learning Repository, dataset "Predict Students' Dropout and Academic Success" (ID 697). Realinho, V., Vieira Martins, M., Machado, J., & Baptista, L. (2021). https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success — DOI: https://doi.org/10.24432/C5MC89
- **Fecha de descarga:** 14 de septiembre de 2026
- **Licencia:** CC BY 4.0 (uso y redistribución permitidos citando la fuente)
- **Unidad de observación:** cada fila es un estudiante matriculado en una carrera de grado de una institución de educación superior portuguesa (varios programas: agronomía, diseño, enfermería, gestión, periodismo, servicio social, tecnologías, entre otros), con cohortes de ingreso entre 2008/09 y 2018/19. El dataset combina datos conocidos al momento de la matrícula (demográficos, socioeconómicos, de admisión) con el desempeño académico al final del 1er y 2do semestre, y el estatus final del estudiante.

**Diccionario breve — variables que usaremos como predictores (información personal y socioeconómica, según la sección 1):**

| Variable | Tipo | Descripción |
|---|---|---|
| `Marital status` | categórica (código) | Estado civil al matricularse |
| `Nacionality` | categórica (código) | Nacionalidad |
| `Displaced` | binaria | Si el estudiante se considera desplazado de su residencia habitual |
| `Educational special needs` | binaria | Si declara necesidades educativas especiales |
| `Debtor` | binaria | Si tiene deudas pendientes con la institución |
| `Tuition fees up to date` | binaria | Si las cuotas de matrícula están al día |
| `Gender` | binaria | Género (codificación original: 0 = femenino, 1 = masculino) |
| `Scholarship holder` | binaria | Si recibe beca |
| `Age at enrollment` | entera | Edad del estudiante al matricularse |
| `International` | binaria | Si es estudiante internacional |
| `Mother's qualification` / `Father's qualification` | categórica (código) | Nivel educativo más alto de la madre / del padre |
| `Mother's occupation` / `Father's occupation` | categórica (código) | Categoría ocupacional de la madre / del padre |

**Variable objetivo derivada:**

| Variable | Tipo | Descripción |
|---|---|---|
| `Target` → `dropout` | binaria (derivada) | 1 si `Target == "Dropout"`, 0 si `Target` es `"Enrolled"` o `"Graduate"` |

Las demás columnas (admisión, desempeño curricular por semestre, indicadores macroeconómicos) se cargan junto con el resto del dataset para el primer barrido, pero quedan fuera de los predictores del modelo por la razón explicada en la sección 1.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42

# Cargar el dataset (descargado de la fuente oficial UCI, ver sección 2)
df = pd.read_csv("data/data.csv", sep=";")
df.columns = df.columns.str.strip()  # el CSV original trae un tab pegado al nombre de una columna

# Variable objetivo binaria definida en la sección 1: 1 = abandonó, 0 = sigue inscrito o se graduó
df["dropout"] = (df["Target"] == "Dropout").astype(int)

df.shape

## 3. Primer barrido

In [ ]:
# df.shape, df.info(), df.head()
print(df.shape)
df.info()
df.head()

In [ ]:
# df.describe() para numéricas, value_counts() para categóricas
df[["Age at enrollment"]].describe()

In [ ]:
# value_counts() de las variables categóricas/personales que usaremos como predictores
personales = ["Marital status", "Nacionality", "Displaced", "Educational special needs",
              "Debtor", "Tuition fees up to date", "Gender", "Scholarship holder", "International"]

for col in personales:
    print(df[col].value_counts().sort_index())
    print()

**Observaciones del primer barrido:**

1. El dataset viene completamente numérico (30 columnas `int64`, 7 `float64`) salvo `Target` (texto). Esto incluye a las variables categóricas que nos interesan (`Marital status`, `Nacionality`, calificación y ocupación de los padres): están codificadas como enteros, no como texto, así que en la sección 6 hay que tratarlas explícitamente como categóricas (`OneHotEncoder`) y no dejar que el pipeline las interprete como numéricas continuas con orden o magnitud.

2. `Nacionality` está extremadamente desbalanceada: 4,314 de 4,424 estudiantes (97.5%) tienen el código 1 (Portugal); el resto se reparte en ~20 categorías con conteos ínfimos (varias con 1-3 estudiantes). Tal cual, esta columna aporta casi cero varianza como predictor y con un one-hot directo generaría columnas casi vacías — candidata a colapsar en "nacional / extranjero" en features.

3. `Marital status` también está concentrada: 88.6% en la categoría 1 (soltero); las categorías 3 y 6 tienen menos de 10 observaciones cada una. Mismo problema de niveles raros que `Nacionality`, aunque menos extremo.

4. Las variables binarias personales (`Displaced`, `Educational special needs`, `Debtor`, `Tuition fees up to date`, `Gender`, `Scholarship holder`, `International`) están limpias, sin valores fuera de {0,1} y sin nulos — no requieren limpieza adicional.

5. `Age at enrollment` tiene media 23.3 y mediana 20 (rango 17-70, desviación estándar 7.6): la distribución tiene cola larga hacia edades mayores (estudiantes que regresan a estudiar), no es simétrica. Vale la pena revisar esa cola en la sección 4 de outliers antes de decidir si se recorta o se deja tal cual.

## 4. Calidad de datos

*Faltantes (y su mecanismo: MCAR, MAR o MNAR, con argumento), duplicados y outliers. Para cada problema: qué hicieron y por qué. Un `dropna()` sin comentario cuesta puntos.*

In [ ]:
# Faltantes por columna, en cantidad y en porcentaje
...

In [ ]:
# Duplicados y outliers (IQR o z-score)
# Para cada outlier detectado: ¿error de captura o señal real?
...

**Decisiones tomadas y su justificación:**

| Problema | Filas o columnas afectadas | Qué hicimos | Por qué |
|---|---|---|---|
|  |  |  |  |

## 5. Análisis exploratorio

*Distribuciones, relaciones y la variable objetivo. Regla del curso: cada gráfica lleva debajo una conclusión escrita. Si no tiene conclusión, la gráfica sobra.*

In [ ]:
# Distribución de la variable objetivo
...

In [ ]:
# Relaciones entre predictores y objetivo (scatter, boxplot por categoría)
...

In [ ]:
# Matriz de correlación (recordar: correlación no es causalidad)
...

**Hallazgos del EDA (numerados, con la gráfica que los respalda):**

1. 
2. 
3. 

## 6. Feature engineering

*Encodings, escalamiento y variables derivadas. Cada transformación necesita una frase que explique por qué existe.*

**Atención al leakage:** la partición train/test va **antes** de ajustar cualquier transformación.

In [ ]:
from sklearn.model_selection import train_test_split

X = ...
y = ...

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE)

In [ ]:
# Transformaciones dentro de un Pipeline: fit solo con train
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preproceso = ...

**Features y su justificación:**

| Feature | Cómo se construyó | Por qué debería ayudar |
|---|---|---|
|  |  |  |

## 7. Modelo baseline

*Primero el baseline trivial (predecir la media o la clase mayoritaria), después el modelo propuesto. Sin el trivial, un número no significa nada.*

In [ ]:
# Baseline trivial: DummyRegressor o DummyClassifier
from sklearn.dummy import DummyRegressor, DummyClassifier

...

In [ ]:
# Modelo propuesto: regresión lineal o k-NN, dentro del Pipeline
...

In [ ]:
# Evaluación en test, con la métrica que corresponda al problema
# Regresión: RMSE, MAE, R2 · Clasificación: accuracy, precision, recall
...

**Resultados:**

| Modelo | Métrica en train | Métrica en test |
|---|---|---|
| Trivial |  |  |
| Propuesto |  |  |

*¿Le gana el modelo propuesto al trivial? ¿Por cuánto? ¿Vale la pena la complejidad extra?*

## 8. Conclusiones y límites

*Respondan la pregunta de la sección 1 con lo que encontraron. Después, un párrafo sobre lo que **no** se puede concluir: sesgos de la muestra, variables ausentes, correlaciones que no son causalidad, y en qué contexto este modelo dejaría de funcionar.*

**Respuesta a la pregunta:**

**Límites del análisis:**

**Qué haría falta para responderla mejor:**

## 9. Bitácora de colaboración con AI

*Según la política del curso: si usaron asistencia de AI, documenten en qué partes, qué les pidieron y qué verificaron ustedes. Si no la usaron, escríbanlo también.*

## Anexo: peer critique (se llena en clase)

**Revisor:**

1. ¿Entendí la pregunta del proyecto sin preguntarle al autor?
2. ¿Hay alguna gráfica o celda que yo borraría? ¿Cuál y por qué?
3. ¿Veo algún punto donde se pudo haber colado información del test?